# División Jumpers

Find players born ≥ 2011 (≤ PREBENJAMÍN age in 2018-2019) who jumped **2+ divisions**
within the same age category between consecutive seasons.

Reference player: **PLATERO SOLIS, AITOR** (id=13794529, born=2014)
- 2024-2025 ALEVÍN: A.C.D. ENTIERGAL — **PRIMERA ALEVIN F-7** (tier 6) → 24 goals
- 2025-2026 ALEVÍN: C.D.A. NAVALCARNERO — **DIVISION DE HONOR ALEVIN** (tier 2) → 2 goals

Tier jump = 6 − 2 = 4, skipping PREFERENTE (4) and PRIMERA DIVISION AUTONOMICA (3) = 2 divisions.

## Output dataframes

| Frame | Rows | Description |
|---|---|---|
| `df_jumps` | one per (player, jump event) | pre/post stats, from/to division |
| `df_career` | one per (player, season) for jumpers | full career timeline |

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('../output/processed/rffm')
SEASONS = sorted(d.name for d in BASE.iterdir() if d.is_dir() and d.name[0].isdigit())

TIER_MAP = {
    'SUPERLIGA': 1,
    'LIGA NACIONAL': 1,
    'DIVISION DE HONOR': 2,
    'PRIMERA DIVISION AUTONOMICA': 3,
    'PREFERENTE': 4,
    'SEGUNDA DIVISION B': 5,
    'TERCERA FEDERACION': 5,
    'PRIMERA': 6,
    'SEGUNDA': 7,
    'TERCERA': 8,
}

# ── Parameters ────────────────────────────────────────────────────────────────
REF_PLAYER_ID   = '13794529'
MIN_BIRTH_YEAR  = 2011   # not older than PREBENJAMÍN age in the 2018-2019 season
MIN_TIER_DIFF   = 4      # PRIMERA(6)→DIVISION DE HONOR(2) = 4 = 2 divisions skipped
EXCLUDE_FEMENINO = True

print('Seasons:', SEASONS)
print(f'MIN_BIRTH_YEAR={MIN_BIRTH_YEAR}  MIN_TIER_DIFF={MIN_TIER_DIFF}')

Seasons: ['2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']
MIN_BIRTH_YEAR=2011  MIN_TIER_DIFF=4


## 1. Reference player career

In [2]:
def _load_ref_participation():
    rows = []
    for s in SEASONS:
        pp = BASE / s / 'player_competition_participation.csv'
        cp = BASE / s / 'competitions.csv'
        if not pp.exists():
            continue
        part = pd.read_csv(pp, dtype=str,
            usecols=['player_id', 'competition_id', 'season', 'team', 'club_name_raw'])
        part = part[part['player_id'] == REF_PLAYER_ID]
        if len(part) == 0:
            continue
        if cp.exists():
            comps = pd.read_csv(cp, dtype=str,
                usecols=['competition_id', 'division_level', 'category_base',
                         'game_type', 'phase_label', 'is_femenino'])
            part = part.merge(comps, on='competition_id', how='left')
        rows.append(part)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def _load_ref_stats():
    rows = []
    for s in SEASONS:
        sp = BASE / s / 'player_season_stats.csv'
        if sp.exists():
            df = pd.read_csv(sp, dtype=str)
            hit = df[df['player_id'] == REF_PLAYER_ID]
            if len(hit):
                rows.append(hit)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


ref_part  = _load_ref_participation()
ref_stats = _load_ref_stats()

if len(ref_part):
    ref_part = ref_part[ref_part['phase_label'] == 'regular_season'].copy()
    ref_part['tier'] = ref_part['division_level'].map(TIER_MAP)
    ref_part = ref_part.dropna(subset=['tier'])
    ref_part['tier'] = ref_part['tier'].astype(int)
    # Best (lowest) tier per season × category
    idx = ref_part.groupby(['season', 'category_base'])['tier'].idxmin()
    ref_best = ref_part.loc[idx].sort_values('season').reset_index(drop=True)

    if len(ref_stats):
        for c in ['matches_played', 'goals_total', 'goals_per_match', 'yellow_cards', 'red_cards']:
            ref_stats[c] = pd.to_numeric(ref_stats[c], errors='coerce')
        ref_best = ref_best.merge(
            ref_stats[['season', 'matches_played', 'goals_total',
                        'goals_per_match', 'yellow_cards', 'red_cards']],
            on='season', how='left'
        )

    print('=== PLATERO SOLIS, AITOR  (id=13794529, born=2014) ===')
    print('Target pattern: jump from PRIMERA (tier 6) to DIVISION DE HONOR (tier 2) in same category\n')
    show = ['season', 'category_base', 'division_level', 'tier', 'game_type',
            'club_name_raw', 'team', 'matches_played', 'goals_total', 'yellow_cards']
    avail = [c for c in show if c in ref_best.columns]
    display(ref_best[avail])
else:
    print('Reference player not found.')

=== PLATERO SOLIS, AITOR  (id=13794529, born=2014) ===
Target pattern: jump from PRIMERA (tier 6) to DIVISION DE HONOR (tier 2) in same category



,season,category_base,division_level,tier,game_type,club_name_raw,team,matches_played,goals_total,yellow_cards
0,2021-2022,PREBENJAMIN,PRIMERA,6,Futbol-7,A.C.D. ENTIERGAL,A.C.D. ENTIERGAL,20,37,0
1,2022-2023,BENJAMIN,PREFERENTE,4,Futbol-7,A.C.D. ENTIERGAL,A.C.D. ENTIERGAL 'A',24,9,0
2,2023-2024,BENJAMIN,PRIMERA,6,Futbol-7,A.C.D. ENTIERGAL,A.C.D. ENTIERGAL 'A',22,61,0
3,2024-2025,ALEVIN,PRIMERA,6,Futbol-7,A.C.D. ENTIERGAL,A.C.D. ENTIERGAL,22,24,0
4,2025-2026,ALEVIN,DIVISION DE HONOR,2,Futbol-11,C.D.A. NAVALCARNERO,C.D.A. NAVALCARNERO 'A',26,2,1
5,2025-2026,INFANTIL,SEGUNDA,7,Futbol-11,C.D.A. NAVALCARNERO,C.D.A. NAVALCARNERO 'F',26,2,1


## 2. Load competition tier data (all seasons)

In [3]:
comps_frames = []
for s in SEASONS:
    cp = BASE / s / 'competitions.csv'
    if cp.exists():
        df = pd.read_csv(cp, dtype=str,
            usecols=['competition_id', 'category_base', 'division_level',
                     'game_type', 'phase_label', 'is_femenino'])
        comps_frames.append(df)

# competition_id is unique per season, so no duplicates across seasons
comps_all = pd.concat(comps_frames, ignore_index=True)
print(f'Total competition rows: {len(comps_all):,}')
print('division_level unique values:', sorted(comps_all['division_level'].dropna().unique()))

Total competition rows: 1,236
division_level unique values: ['CAMPEONATO UNIVERSITARIO', 'DIVISION DE HONOR', 'FASE ZONAL', 'LIGA NACIONAL', 'LIGA UNIVERSITARIA', 'OTHER', 'PREFERENTE', 'PRIMERA', 'PRIMERA DIVISION AUTONOMICA', 'SEGUNDA', 'SEGUNDA DIVISION B', 'SUPERLIGA', 'TERCERA', 'TERCERA FEDERACION']


## 3. Load player participation + enrich with tier

For each `(player_id, season, category_base)` keep only the **best (lowest) tier** row.
Filters applied: `phase_label == 'regular_season'`, optionally `is_femenino != True`.

In [4]:
part_frames = []
for s in SEASONS:
    pp = BASE / s / 'player_competition_participation.csv'
    if pp.exists():
        df = pd.read_csv(pp, dtype=str,
            usecols=['player_id', 'competition_id', 'season', 'team', 'club_name_raw'])
        part_frames.append(df)

part_all = pd.concat(part_frames, ignore_index=True)
print(f'Total participation rows: {len(part_all):,}')

# Enrich with competition metadata
part_rich = part_all.merge(
    comps_all[['competition_id', 'category_base', 'division_level',
               'game_type', 'phase_label', 'is_femenino']],
    on='competition_id', how='left'
)

# Keep regular season only (cups/playoffs share division_level with parent league)
part_rich = part_rich[part_rich['phase_label'] == 'regular_season']

if EXCLUDE_FEMENINO:
    part_rich = part_rich[part_rich['is_femenino'].str.lower() != 'true']

# Map to tier; drop unranked (OTHER, FASE ZONAL, UNIVERSITARIO, etc.)
part_rich = part_rich.copy()
part_rich['tier'] = part_rich['division_level'].map(TIER_MAP)
part_rich = part_rich.dropna(subset=['tier'])
part_rich['tier'] = part_rich['tier'].astype(int)

print(f'After filters: {len(part_rich):,} rows')

# Best (lowest) tier per (player_id, season, category_base)
idx = part_rich.groupby(['player_id', 'season', 'category_base'])['tier'].idxmin()
df_part = (
    part_rich.loc[idx,
        ['player_id', 'season', 'category_base',
         'division_level', 'tier', 'game_type', 'club_name_raw', 'team']]
    .reset_index(drop=True)
)

print(f'Unique (player, season, category) entries: {len(df_part):,}')
df_part.head(3)

Total participation rows: 1,247,419
After filters: 1,039,005 rows
Unique (player, season, category) entries: 916,672


,player_id,season,category_base,division_level,tier,game_type,club_name_raw,team
0,10000023,2019-2020,ALEVIN,PREFERENTE,4,Futbol-11,C.D.E. F.P.A. LAS ROZAS,C.D.E. F.P.A. LAS ROZAS 'A'
1,10000023,2020-2021,INFANTIL,PREFERENTE,4,Futbol-11,C.D.E. F.P.A. LAS ROZAS,C.D.E. F.P.A. LAS ROZAS 'A'
2,10000023,2021-2022,CADETE,PREFERENTE,4,Futbol-11,C.D.E. F.P.A. LAS ROZAS,C.D.E. F.P.A. LAS ROZAS 'A'


## 4. Load player stats & identities

In [5]:
# ── Season stats (one row per player per season, site-reported aggregated totals) ──
stats_frames = []
for s in SEASONS:
    sp = BASE / s / 'player_season_stats.csv'
    if sp.exists():
        df = pd.read_csv(sp, dtype=str,
            usecols=['player_id', 'season',
                     'matches_played', 'goals_total', 'goals_per_match',
                     'yellow_cards', 'red_cards',
                     'called_up', 'starter_appearances'])
        stats_frames.append(df)

df_stats = pd.concat(stats_frames, ignore_index=True)
for c in ['matches_played', 'goals_total', 'yellow_cards', 'red_cards',
          'called_up', 'starter_appearances']:
    df_stats[c] = pd.to_numeric(df_stats[c], errors='coerce')
df_stats['goals_per_match'] = pd.to_numeric(df_stats['goals_per_match'], errors='coerce').round(2)

# If a player somehow has duplicate rows in one season file, sum stats
df_stats = df_stats.groupby(['player_id', 'season'], as_index=False).agg(
    matches_played=('matches_played', 'sum'),
    goals_total=('goals_total', 'sum'),
    goals_per_match=('goals_per_match', 'mean'),
    yellow_cards=('yellow_cards', 'sum'),
    red_cards=('red_cards', 'sum'),
    called_up=('called_up', 'sum'),
    starter_appearances=('starter_appearances', 'sum'),
)

print(f'Stats rows: {len(df_stats):,}')

# ── Player identities (birth_year) ────────────────────────────────────────────
players_frames = []
for s in SEASONS:
    pp = BASE / s / 'players.csv'
    if pp.exists():
        df = pd.read_csv(pp, dtype=str,
            usecols=['player_id', 'player_name', 'birth_year'])
        df['_season'] = s
        players_frames.append(df)

players_all = pd.concat(players_frames, ignore_index=True)
df_players = (
    players_all.sort_values('_season')
    .drop_duplicates(subset='player_id', keep='last')
    [['player_id', 'player_name', 'birth_year']]
    .reset_index(drop=True)
)
df_players['birth_year'] = pd.to_numeric(df_players['birth_year'], errors='coerce')

print(f'Unique players: {len(df_players):,}')

Stats rows: 908,400
Unique players: 289,346


## 5. Detect division jumps

For each `(player_id, category_base)`, compare consecutive seasons (sorted by season string).
A **jump** is a `tier_diff = prev_tier − current_tier ≥ MIN_TIER_DIFF` (moved up by ≥ 4 tiers).

Raw tier_diff ≥ 4 in youth football (where tier 5 is adult-only) corresponds to skipping ≥ 2 intermediate divisions.

In [6]:
df_sorted = df_part.sort_values(['player_id', 'category_base', 'season']).copy()

grp = df_sorted.groupby(['player_id', 'category_base'])

df_sorted['prev_season']    = grp['season'].shift(1)
df_sorted['prev_tier']      = grp['tier'].shift(1)
df_sorted['prev_division']  = grp['division_level'].shift(1)
df_sorted['prev_game_type'] = grp['game_type'].shift(1)
df_sorted['prev_club']      = grp['club_name_raw'].shift(1)
df_sorted['prev_team']      = grp['team'].shift(1)

# Keep only rows with a prior season in the same category
df_shifted = df_sorted.dropna(subset=['prev_season', 'prev_tier']).copy()
df_shifted['prev_tier'] = df_shifted['prev_tier'].astype(int)
df_shifted['tier_diff'] = df_shifted['prev_tier'] - df_shifted['tier']

# Select jumps
df_jumps_raw = df_shifted[df_shifted['tier_diff'] >= MIN_TIER_DIFF].copy()
df_jumps_raw = df_jumps_raw.rename(columns={
    'season':        'jump_to_season',
    'tier':          'to_tier',
    'division_level':'to_division',
    'game_type':     'to_game_type',
    'club_name_raw': 'to_club',
    'team':          'to_team',
    'prev_season':   'jump_from_season',
    'prev_tier':     'from_tier',
    'prev_division': 'from_division',
    'prev_game_type':'from_game_type',
    'prev_club':     'from_club',
    'prev_team':     'from_team',
})

cols_keep = ['player_id', 'category_base',
             'jump_from_season', 'from_division', 'from_tier', 'from_game_type', 'from_club', 'from_team',
             'jump_to_season',   'to_division',   'to_tier',   'to_game_type',   'to_club',   'to_team',
             'tier_diff']
df_jumps_raw = df_jumps_raw[cols_keep]

print(f'Raw jump events (all players, all birth years): {len(df_jumps_raw):,}')
print('\nTop jumps by tier_diff:')
display(df_jumps_raw.sort_values('tier_diff', ascending=False).head(10))

Raw jump events (all players, all birth years): 9,384

Top jumps by tier_diff:


,player_id,category_base,jump_from_season,from_division,from_tier,from_game_type,from_club,from_team,jump_to_season,to_division,to_tier,to_game_type,to_club,to_team,tier_diff
241290,14501452,INFANTIL,2022-2023,SEGUNDA,7,Futbol-11,C.F. ALALPARDO,C.F. ALALPARDO 'D',2023-2024,SUPERLIGA,1,Futbol-11,C.D. CHAMARTIN VERGARA,C.D. CHAMARTIN VERGARA - ALCOBENDAS 'A',6
411086,20517078,ALEVIN,2023-2024,SEGUNDA,7,Futbol-7,MORALZARZAL C.F.,MORALZARZAL C.F. 'C',2024-2025,SUPERLIGA,1,Futbol-11,LAS ROZAS C.F.,LAS ROZAS C.F. 'A',6
410956,20514810,CADETE,2024-2025,SEGUNDA,7,Futbol-11,SESEÑA C.F.,SESEÑA C.F. 'D',2025-2026,SUPERLIGA,1,Futbol-11,SESEÑA C.F.,SESEÑA C.F. 'A',6
680462,4589814,JUVENIL,2024-2025,SEGUNDA,7,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'C',2025-2026,LIGA NACIONAL,1,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'A',6
212329,13890986,INFANTIL,2023-2024,SEGUNDA,7,Futbol-11,A.D. TORREJON C.F.,A.D. TORREJON C.F. 'E',2024-2025,SUPERLIGA,1,Futbol-11,SAD COLEGIO MIRAMADRID,COLEGIO MIRAMADRID - PARACUELLOS 'A',6
290875,15462654,INFANTIL,2023-2024,SEGUNDA,7,Futbol-11,S.A.D. A.D.C. PARQUE SURESTE,S.A.D. A.D.C. PARQUE SURESTE 'E',2024-2025,SUPERLIGA,1,Futbol-11,S.A.D. FUNDACION RAYO VALLECANO,S.A.D. FUNDACION RAYO VALLECANO 'A',6
788962,7767313,INFANTIL,2021-2022,SEGUNDA,7,Futbol-11,ESCUELA DEPORTIVA MORATALAZ,ESCUELA DEP. MORATALAZ 'F',2022-2023,SUPERLIGA,1,Futbol-11,ESCUELA DEPORTIVA MORATALAZ,ESCUELA DEP. MORATALAZ 'A',6
291724,15494825,CADETE,2021-2022,SEGUNDA,7,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'E',2022-2023,SUPERLIGA,1,Futbol-11,REAL MADRID C.F.,REAL MADRID C.F. 'A',6
211333,13860294,INFANTIL,2023-2024,SEGUNDA,7,Futbol-11,C.D.A. NAVALCARNERO,C.D.A. NAVALCARNERO 'G',2024-2025,SUPERLIGA,1,Futbol-11,A.D. ALCORCON S.A.D.,A.D. ALCORCON S.A.D. 'A',6
211331,13860294,ALEVIN,2022-2023,SEGUNDA,7,Futbol-7,C.D.A. NAVALCARNERO,C.D.A. NAVALCARNERO 'E',2023-2024,SUPERLIGA,1,Futbol-11,C.D. NUEVO BOADILLA,C.D. NUEVO BOADILLA 'A',6


## 6. Filter cohort + build `df_jumps`

Keep only players with `birth_year ≥ MIN_BIRTH_YEAR` (= not older than PREBENJAMÍN age in 2018-2019).
Add pre-jump and post-jump season stats.

In [7]:
# Add player identity
df_jumps_raw = df_jumps_raw.merge(df_players, on='player_id', how='left')

# Cohort filter
df_jumps_cohort = df_jumps_raw[
    df_jumps_raw['birth_year'].notna() &
    (df_jumps_raw['birth_year'] >= MIN_BIRTH_YEAR)
].copy()

print(f'After birth_year >= {MIN_BIRTH_YEAR}: '
      f'{len(df_jumps_cohort):,} jump events, '
      f'{df_jumps_cohort["player_id"].nunique():,} unique players')

# First season per player
first_season = (
    df_part.groupby('player_id')['season']
    .min()
    .rename('first_season')
    .reset_index()
)
df_jumps_cohort = df_jumps_cohort.merge(first_season, on='player_id', how='left')

# Pre-jump stats (season before the jump)
pre = df_stats.rename(columns={
    'season':              'jump_from_season',
    'matches_played':      'pre_matches',
    'goals_total':         'pre_goals',
    'goals_per_match':     'pre_goals_per_match',
    'yellow_cards':        'pre_yellow',
    'red_cards':           'pre_red',
    'called_up':           'pre_called_up',
    'starter_appearances': 'pre_starters',
})
df_jumps_cohort = df_jumps_cohort.merge(
    pre[['player_id', 'jump_from_season',
         'pre_matches', 'pre_goals', 'pre_goals_per_match', 'pre_yellow', 'pre_red']],
    on=['player_id', 'jump_from_season'], how='left'
)

# Post-jump stats (the jump season itself)
post = df_stats.rename(columns={
    'season':              'jump_to_season',
    'matches_played':      'post_matches',
    'goals_total':         'post_goals',
    'goals_per_match':     'post_goals_per_match',
    'yellow_cards':        'post_yellow',
    'red_cards':           'post_red',
    'called_up':           'post_called_up',
    'starter_appearances': 'post_starters',
})
df_jumps_cohort = df_jumps_cohort.merge(
    post[['player_id', 'jump_to_season',
          'post_matches', 'post_goals', 'post_goals_per_match', 'post_yellow', 'post_red']],
    on=['player_id', 'jump_to_season'], how='left'
)

# Final column order
df_jumps = df_jumps_cohort[[
    'player_id', 'player_name', 'birth_year', 'first_season',
    'category_base',
    'jump_from_season', 'from_division', 'from_tier', 'from_game_type', 'from_club', 'from_team',
    'jump_to_season',   'to_division',   'to_tier',   'to_game_type',   'to_club',   'to_team',
    'tier_diff',
    'pre_matches',  'pre_goals',  'pre_goals_per_match',  'pre_yellow',  'pre_red',
    'post_matches', 'post_goals', 'post_goals_per_match', 'post_yellow', 'post_red',
]].sort_values(['tier_diff', 'jump_to_season'], ascending=[False, True]).reset_index(drop=True)

print(f'\ndf_jumps: {len(df_jumps):,} rows, {df_jumps["player_id"].nunique():,} unique players')
df_jumps.head(10)

After birth_year >= 2011: 3,638 jump events, 3,527 unique players

df_jumps: 3,638 rows, 3,527 unique players


,player_id,player_name,birth_year,first_season,category_base,jump_from_season,from_division,from_tier,from_game_type,from_club,...,pre_matches,pre_goals,pre_goals_per_match,pre_yellow,pre_red,post_matches,post_goals,post_goals_per_match,post_yellow,post_red
0,10150601,"AITKENHEAD ARAUJO, LUCAS",2012.0,2018-2019,ALEVIN,2021-2022,SEGUNDA,7,Futbol-7,REEBOK SPORTS CLUB ACADEMIA DE FUTBOL,...,28,80,2.9,0,0,25,9,0.4,0,0
1,10166745,"CRISTOBAL ARGUELLO, YAGO",2011.0,2019-2020,ALEVIN,2021-2022,SEGUNDA,7,Futbol-7,RAYO CIUDAD ALCOBENDAS C.F.,...,20,16,0.8,0,0,26,1,0.0,0,0
2,10762389,"RODRIGUEZ TORRES, DANIEL",2012.0,2020-2021,ALEVIN,2022-2023,SEGUNDA,7,Futbol-7,C.D.E. FOOTBALL DREAMS EXPERIENCE,...,27,52,1.9,0,0,23,2,0.1,1,0
3,11359708,"CARPIO FERNANDEZ, MARC ANTHONY",2012.0,2020-2021,ALEVIN,2022-2023,SEGUNDA,7,Futbol-7,C.D.E. DV7 - MADRID,...,22,21,1.0,0,0,25,0,0.0,2,0
4,11884920,"MARTINEZ GARCIA, ENZO",2013.0,2020-2021,ALEVIN,2021-2022,SEGUNDA,7,Futbol-7,C.D. SP&AL SERRANILLOS,...,27,0,0.0,0,0,28,0,0.0,0,0
5,11925317,"ALARCON CRIADO, ANDRE FERNANDO",2012.0,2020-2021,ALEVIN,2022-2023,SEGUNDA,7,Futbol-7,S.A.D. FUNDACION RAYO MAJADAHONDA,...,18,10,0.6,0,0,19,1,0.1,1,0
6,12264144,"ARIAS PESQUERO, PABLO",2012.0,2020-2021,ALEVIN,2022-2023,SEGUNDA,7,Futbol-7,GETAFE C.F. S.A.D.,...,22,36,1.6,0,0,28,1,0.0,0,0
7,12264294,"GOMEZ GARCIA, GUILLERMO",2012.0,2020-2021,ALEVIN,2022-2023,SEGUNDA,7,Futbol-7,GETAFE C.F. S.A.D.,...,18,5,0.3,0,0,24,2,0.1,0,0
8,13177581,"FUENZALIDA MUÑOZ, DIEGO",2012.0,2020-2021,ALEVIN,2022-2023,SEGUNDA,7,Futbol-7,C.D. NUEVO BOADILLA,...,20,0,0.0,0,0,24,0,0.0,0,0
9,13184127,"GUTIERREZ MONTANER, RODRIGO",2013.0,2020-2021,ALEVIN,2022-2023,SEGUNDA,7,Futbol-7,C.F. FUENLABRADA S.A.D.,...,22,0,0.0,0,0,24,0,0.0,0,0


## 7. Full career table for jump players (`df_career`)

One row per `(player_id, season)` for every player in `df_jumps`.
`is_jump_season = True` marks the season in which the division jump happened.

In [8]:
jumper_ids = df_jumps['player_id'].unique()

career_part  = df_part[df_part['player_id'].isin(jumper_ids)].copy()
career_stats = df_stats[df_stats['player_id'].isin(jumper_ids)].copy()

# is_jump_season flag
jump_flags = (
    df_jumps[['player_id', 'jump_to_season']]
    .drop_duplicates()
    .rename(columns={'jump_to_season': 'season'})
    .assign(is_jump_season=True)
)

# Merge participation + stats
df_career = career_part.merge(
    career_stats[['player_id', 'season',
                  'matches_played', 'goals_total', 'goals_per_match',
                  'yellow_cards', 'red_cards', 'called_up', 'starter_appearances']],
    on=['player_id', 'season'], how='left'
)

# Add player name + birth_year
df_career = df_career.merge(df_players, on='player_id', how='left')

# Add jump flag
df_career = df_career.merge(jump_flags, on=['player_id', 'season'], how='left')
df_career['is_jump_season'] = df_career['is_jump_season'].fillna(False)

df_career = df_career[[
    'player_id', 'player_name', 'birth_year',
    'season', 'category_base', 'division_level', 'tier', 'game_type',
    'club_name_raw', 'team',
    'matches_played', 'goals_total', 'goals_per_match',
    'yellow_cards', 'red_cards', 'called_up', 'starter_appearances',
    'is_jump_season',
]].sort_values(['player_id', 'season']).reset_index(drop=True)

print(f'df_career: {len(df_career):,} rows for {df_career["player_id"].nunique():,} players')
df_career.head(5)

df_career: 18,984 rows for 3,527 players


C:\Users\gshushuev\AppData\Local\Temp\ipykernel_4048\225199580.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_career['is_jump_season'] = df_career['is_jump_season'].fillna(False)


,player_id,player_name,birth_year,season,category_base,division_level,tier,game_type,club_name_raw,team,matches_played,goals_total,goals_per_match,yellow_cards,red_cards,called_up,starter_appearances,is_jump_season
0,10004840,"MASALAB, DANYLO",2011.0,2021-2022,ALEVIN,PRIMERA,6,Futbol-7,CLUB FUENTELARREYNA,CLUB FUENTELARREYNA 'B',13,0,0.0,0,0,13,8,False
1,10004840,"MASALAB, DANYLO",2011.0,2022-2023,ALEVIN,SUPERLIGA,1,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'A',26,0,0.0,0,0,26,12,True
2,10004840,"MASALAB, DANYLO",2011.0,2022-2023,INFANTIL,SEGUNDA,7,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'E',26,0,0.0,0,0,26,12,True
3,10004840,"MASALAB, DANYLO",2011.0,2023-2024,INFANTIL,PRIMERA DIVISION AUTONOMICA,3,Futbol-11,RAYO CIUDAD ALCOBENDAS C.F.,RAYO CIUDAD ALCOBENDAS C.F. 'B',22,0,0.0,0,0,22,3,True
4,10004840,"MASALAB, DANYLO",2011.0,2024-2025,INFANTIL,DIVISION DE HONOR,2,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'A',19,0,0.0,0,0,19,2,False


## 8. Summary statistics

In [9]:
print('=' * 60)
print(f'Total jump events : {len(df_jumps):,}')
print(f'Unique players    : {df_jumps["player_id"].nunique():,}')
by = df_jumps['birth_year'].dropna().astype(int)
print(f'Birth year range  : {by.min()} – {by.max()}')

print('\nJumps by tier_diff:')
print(df_jumps['tier_diff'].value_counts().sort_index().to_string())

print('\nJumps by category_base:')
print(df_jumps['category_base'].value_counts().to_string())

print('\nTop 15 division transitions:')
df_jumps['transition'] = df_jumps['from_division'] + '  →  ' + df_jumps['to_division']
print(df_jumps['transition'].value_counts().head(15).to_string())

print('\n=== Most dramatic jumps (highest tier_diff) ===')
top = df_jumps[['player_name', 'birth_year', 'category_base',
                'jump_from_season', 'from_division',
                'jump_to_season', 'to_division',
                'tier_diff',
                'pre_goals', 'pre_goals_per_match',
                'post_goals', 'post_goals_per_match']].head(30)
display(top)

Total jump events : 3,638
Unique players    : 3,527
Birth year range  : 2011 – 2017

Jumps by tier_diff:
tier_diff
4    2810
5     772
6      56

Jumps by category_base:
category_base
ALEVIN      2099
INFANTIL    1233
BENJAMIN     216
CADETE        90

Top 15 division transitions:
transition
SEGUNDA  →  PRIMERA DIVISION AUTONOMICA    1804
PRIMERA  →  DIVISION DE HONOR              1006
SEGUNDA  →  DIVISION DE HONOR               641
PRIMERA  →  SUPERLIGA                       131
SEGUNDA  →  SUPERLIGA                        56

=== Most dramatic jumps (highest tier_diff) ===


,player_name,birth_year,category_base,jump_from_season,from_division,jump_to_season,to_division,tier_diff,pre_goals,pre_goals_per_match,post_goals,post_goals_per_match
0,"AITKENHEAD ARAUJO, LUCAS",2012.0,ALEVIN,2021-2022,SEGUNDA,2022-2023,SUPERLIGA,6,80,2.9,9,0.4
1,"CRISTOBAL ARGUELLO, YAGO",2011.0,ALEVIN,2021-2022,SEGUNDA,2022-2023,SUPERLIGA,6,16,0.8,1,0.0
2,"RODRIGUEZ TORRES, DANIEL",2012.0,ALEVIN,2022-2023,SEGUNDA,2023-2024,SUPERLIGA,6,52,1.9,2,0.1
3,"CARPIO FERNANDEZ, MARC ANTHONY",2012.0,ALEVIN,2022-2023,SEGUNDA,2023-2024,SUPERLIGA,6,21,1.0,0,0.0
4,"MARTINEZ GARCIA, ENZO",2013.0,ALEVIN,2021-2022,SEGUNDA,2023-2024,SUPERLIGA,6,0,0.0,0,0.0
5,"ALARCON CRIADO, ANDRE FERNANDO",2012.0,ALEVIN,2022-2023,SEGUNDA,2023-2024,SUPERLIGA,6,10,0.6,1,0.1
6,"ARIAS PESQUERO, PABLO",2012.0,ALEVIN,2022-2023,SEGUNDA,2023-2024,SUPERLIGA,6,36,1.6,1,0.0
7,"GOMEZ GARCIA, GUILLERMO",2012.0,ALEVIN,2022-2023,SEGUNDA,2023-2024,SUPERLIGA,6,5,0.3,2,0.1
8,"FUENZALIDA MUÑOZ, DIEGO",2012.0,ALEVIN,2022-2023,SEGUNDA,2023-2024,SUPERLIGA,6,0,0.0,0,0.0
9,"GUTIERREZ MONTANER, RODRIGO",2013.0,ALEVIN,2022-2023,SEGUNDA,2023-2024,SUPERLIGA,6,0,0.0,0,0.0


## 9. Per-player career tables

Full career timeline for each jump player. Jump season rows are highlighted.

In [14]:
aravaca_ids = df_career.loc[
    df_career['club_name_raw'].str.contains('aravaca', case=False, na=False),
    'player_id'
].unique()

df_career[df_career['player_id'].isin(aravaca_ids)].reset_index(drop=True).to_excel('../output/notebooks/division_jumpers_aravaca.xlsx', index=False)

In [15]:
show_cols = ['season', 'category_base', 'division_level', 'tier', 'game_type',
             'club_name_raw', 'team',
             'matches_played', 'goals_total', 'goals_per_match',
             'yellow_cards', 'is_jump_season']

limit = 10
for i, (player_id, group) in enumerate(df_career.groupby('player_id')):
    if i >= limit:
        break
    player_name = group['player_name'].iloc[0]
    birth_year  = group['birth_year'].iloc[0]
    by_str = str(int(birth_year)) if pd.notna(birth_year) else '?'

    player_jumps = df_jumps[df_jumps['player_id'] == player_id]

    print(f'\n{"=" * 68}')
    print(f'{player_name}  (id={player_id}, born={by_str})')
    for _, jrow in player_jumps.iterrows():
        pre_g  = jrow['pre_goals']  if pd.notna(jrow['pre_goals'])  else '?'
        post_g = jrow['post_goals'] if pd.notna(jrow['post_goals']) else '?'
        print(f'  JUMP [{jrow["category_base"]}]: '
              f'{jrow["jump_from_season"]} {jrow["from_division"]} (tier {jrow["from_tier"]}, {pre_g} goles)'
              f'  →  {jrow["jump_to_season"]} {jrow["to_division"]} (tier {jrow["to_tier"]}, {post_g} goles)')

    avail = [c for c in show_cols if c in group.columns]
    tbl   = group[avail].reset_index(drop=True)

    def highlight_jump(row):
        color = 'background-color: #ffe066' if row.get('is_jump_season', False) else ''
        return [color] * len(row)

    display(tbl.style.apply(highlight_jump, axis=1))


MASALAB, DANYLO  (id=10004840, born=2011)
  JUMP [ALEVIN]: 2021-2022 PRIMERA (tier 6, 0 goles)  →  2022-2023 SUPERLIGA (tier 1, 0 goles)
  JUMP [INFANTIL]: 2022-2023 SEGUNDA (tier 7, 0 goles)  →  2023-2024 PRIMERA DIVISION AUTONOMICA (tier 3, 0 goles)


,season,category_base,division_level,tier,game_type,club_name_raw,team,matches_played,goals_total,goals_per_match,yellow_cards,is_jump_season
0,2021-2022,ALEVIN,PRIMERA,6,Futbol-7,CLUB FUENTELARREYNA,CLUB FUENTELARREYNA 'B',13,0,0.000000,0,False
1,2022-2023,ALEVIN,SUPERLIGA,1,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'A',26,0,0.000000,0,True
2,2022-2023,INFANTIL,SEGUNDA,7,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'E',26,0,0.000000,0,True
3,2023-2024,INFANTIL,PRIMERA DIVISION AUTONOMICA,3,Futbol-11,RAYO CIUDAD ALCOBENDAS C.F.,RAYO CIUDAD ALCOBENDAS C.F. 'B',22,0,0.000000,0,True
4,2024-2025,INFANTIL,DIVISION DE HONOR,2,Futbol-11,ATLETICO CHOPERA ALCOBENDAS 04,ATLETICO CHOPERA ALCOBENDAS 04 'A',19,0,0.000000,0,False
5,2025-2026,CADETE,PRIMERA,6,Futbol-11,A.D. COLMENAR VIEJO,A.D. COLMENAR VIEJO 'B',29,0,0.000000,0,False



LOUHAMANE RUIZ, OMAR  (id=10007223, born=2012)
  JUMP [INFANTIL]: 2024-2025 SEGUNDA (tier 7, 6 goles)  →  2025-2026 DIVISION DE HONOR (tier 2, 0 goles)


,season,category_base,division_level,tier,game_type,club_name_raw,team,matches_played,goals_total,goals_per_match,yellow_cards,is_jump_season
0,2020-2021,BENJAMIN,PRIMERA,6,Futbol-7,A.D.C. SAN FERMIN,A.D.C. SAN FERMIN 'B',10,0,0.000000,0,False
1,2022-2023,ALEVIN,SEGUNDA,7,Futbol-7,ASOC. CULTURAL EL BERCIAL,ASOC. CULTURAL EL BERCIAL 'B',28,7,0.300000,1,False
2,2023-2024,ALEVIN,PRIMERA,6,Futbol-7,ASOC. CULTURAL EL BERCIAL,ASOC. CULTURAL EL BERCIAL 'A',24,1,0.000000,0,False
3,2023-2024,INFANTIL,SEGUNDA,7,Futbol-11,ASOC. CULTURAL EL BERCIAL,ASOC. CULTURAL EL BERCIAL,24,1,0.000000,0,False
4,2024-2025,INFANTIL,SEGUNDA,7,Futbol-11,ATLETICO CLUB DE SOCIOS,ATLETICO CLUB DE SOCIOS - BERCIAL,26,6,0.200000,3,False
5,2025-2026,CADETE,PRIMERA,6,Futbol-11,CDE INTER PROMESAS,CDE INTER PROMESAS 'C',28,0,0.000000,2,True
6,2025-2026,INFANTIL,DIVISION DE HONOR,2,Futbol-11,CDE INTER PROMESAS,CDE INTER PROMESAS 'A',28,0,0.000000,2,True



SANCHEZ MOLINA, BERNA  (id=10007636, born=2012)
  JUMP [INFANTIL]: 2024-2025 PRIMERA (tier 6, 10 goles)  →  2025-2026 DIVISION DE HONOR (tier 2, 13 goles)


...

,season,category_base,division_level,tier,game_type,club_name_raw,team,matches_played,goals_total,goals_per_match,yellow_cards,is_jump_season
0,2019-2020,BENJAMIN,PRIMERA,6,Futbol-7,C.F.D. ELIDA OLIMPIA,C.F.D. ELIDA OLIMPIA 'C',14,4,0.300000,0,False
1,2020-2021,BENJAMIN,PREFERENTE,4,Futbol-7,C.F.D. ELIDA OLIMPIA,C.F.D. ELIDA OLIMPIA 'A',12,0,0.000000,0,False
2,2021-2022,ALEVIN,PRIMERA,6,Futbol-11,C.F.D. ELIDA OLIMPIA,C.F.D. ELIDA OLIMPIA 'B',25,8,0.300000,0,False
3,2021-2022,BENJAMIN,PRIMERA DIVISION AUTONOMICA,3,Futbol-7,C.F.D. ELIDA OLIMPIA,C.F.D. ELIDA OLIMPIA 'A',25,8,0.300000,0,False
4,2022-2023,ALEVIN,DIVISION DE HONOR,2,Futbol-7,S.A.D. FUNDACION RAYO VALLECANO,S.A.D. FUNDACION RAYO VALLECANO 'A',25,1,0.000000,3,True
5,2023-2024,ALEVIN,DIVISION DE HONOR,2,Futbol-11,S.A.D. FUNDACION RAYO VALLECANO,S.A.D. FUNDACION RAYO VALLECANO 'A',28,6,0.200000,10,False
6,2024-2025,INFANTIL,PREFERENTE,4,Futbol-11,S.A.D. FUNDACION RAYO VALLECANO,S.A.D. FUNDACION RAYO VALLECANO 'C',27,2,0.100000,2,False
7,2025-2026,INFANTIL,DIVISION DE HONOR,2,Futbol-11,S.A.D. FUNDACION RAYO VALLECANO,S.A.D. FUNDACION RAYO VALLECANO 'A',29,4,0.100000,6,False



IGLESIAS GARCIA, ILLAN  (id=10016590, born=2012)
  JUMP [INFANTIL]: 2024-2025 PRIMERA (tier 6, 5 goles)  →  2025-2026 DIVISION DE HONOR (tier 2, 0 goles)


,season,category_base,division_level,tier,game_type,club_name_raw,team,matches_played,goals_total,goals_per_match,yellow_cards,is_jump_season
0,2020-2021,BENJAMIN,PRIMERA,6,Futbol-7,C.D. MADRID SUR LATINA,C.D. MADRID SUR LATINA,12,2,0.200000,0,False
1,2021-2022,ALEVIN,PRIMERA,6,Futbol-11,C.D. MADRID SUR LATINA,C.D. MADRID SUR LATINA 'B',25,18,0.700000,0,False
2,2021-2022,BENJAMIN,PRIMERA,6,Futbol-7,C.D. MADRID SUR LATINA,C.D. MADRID SUR LATINA,25,18,0.700000,0,False
3,2022-2023,ALEVIN,SEGUNDA,7,Futbol-7,C.D. MADRID SUR LATINA,C.D. MADRID SUR LATINA 'A',22,3,0.100000,0,False
4,2023-2024,ALEVIN,PRIMERA,6,Futbol-11,C.D. MADRID SUR LATINA,C.D. MADRID SUR LATINA,26,5,0.200000,0,False
5,2024-2025,INFANTIL,PRIMERA,6,Futbol-11,C.D. MADRID SUR LATINA,C.D. MADRID SUR LATINA 'A',29,5,0.200000,2,False
6,2025-2026,INFANTIL,DIVISION DE HONOR,2,Futbol-11,A.D. UNION CARRASCAL,A.D. UNION CARRASCAL 'A',26,0,0.000000,4,True



BELTRE ZAMORANO, CRISTIAN  (id=10018408, born=2012)
  JUMP [INFANTIL]: 2024-2025 SEGUNDA (tier 7, 0 goles)  →  2025-2026 DIVISION DE HONOR (tier 2, 1 goles)


,season,category_base,division_level,tier,game_type,club_name_raw,team,matches_played,goals_total,goals_per_match,yellow_cards,is_jump_season
0,2021-2022,BENJAMIN,PRIMERA,6,Futbol-7,SAN PEDRO DE HUMANES,SAN PEDRO DE HUMANES 'C',18,1,0.100000,0,False
1,2022-2023,ALEVIN,PREFERENTE,4,Futbol-7,SAN PEDRO DE HUMANES,SAN PEDRO DE HUMANES,27,0,0.000000,0,False
2,2023-2024,ALEVIN,PRIMERA,6,Futbol-11,SAN PEDRO DE HUMANES,SAN PEDRO DE HUMANES 'A',22,0,0.000000,0,False
3,2024-2025,INFANTIL,SEGUNDA,7,Futbol-11,SAN PEDRO DE HUMANES,SAN PEDRO DE HUMANES 'B',23,0,0.000000,0,False
4,2025-2026,CADETE,SEGUNDA,7,Futbol-11,S.A.D. A.D.C. PARQUE SURESTE,S.A.D. A.D.C. PARQUE SURESTE 'D',27,1,0.000000,3,True
5,2025-2026,INFANTIL,DIVISION DE HONOR,2,Futbol-11,S.A.D. A.D.C. PARQUE SURESTE,S.A.D. A.D.C. PARQUE SURESTE 'A',27,1,0.000000,3,True


## 10. Export

In [16]:
out_jumps  = BASE / 'career_analysis_out_jumps.csv'
out_career = BASE / 'career_analysis_out_career.csv'

df_out = df_jumps.drop(columns=['transition'], errors='ignore')
df_out.to_csv(out_jumps, index=False)
df_career.to_csv(out_career, index=False)

print(f'df_jumps  → {out_jumps}  ({len(df_out):,} rows)')
print(f'df_career → {out_career}  ({len(df_career):,} rows)')

df_jumps  → ..\output\processed\rffm\career_analysis_out_jumps.csv  (3,638 rows)
df_career → ..\output\processed\rffm\career_analysis_out_career.csv  (18,984 rows)
